# citegraph quickstart

Two ways to run the pipeline on a folder of PDFs:

1. **All at once** — `pipe.run()` does every stage end-to-end.
2. **Step by step** — call each stage method, inspect the artifact on disk, decide whether to continue.

Both modes share the same `out_dir`, so you can mix and match: kick off `run()` once and then re-run an individual stage to iterate on it.

Set `GOOGLE_API_KEY` in your environment (or in a `.env` file in the current working directory) before running the cells below.

In [ ]:
from pathlib import Path

import pandas as pd

from citegraph import Pipeline

PDF_DIR = Path("../../data/pdfs")  # adapt to your project layout
OUT_DIR = Path("../out")

pipe = Pipeline(pdf_dir=PDF_DIR, out_dir=OUT_DIR, enrich=False)

## Mode A — run the whole pipeline

Use this when you trust the defaults and just want the three CSVs.
Skip this section entirely if you'd rather drive the pipeline stage-by-stage in Mode B.

In [ ]:
result = pipe.run()
print(
    f"papers={len(result.papers)}, "
    f"references={len(result.references)}, "
    f"edges={len(result.graph)}"
)

In [ ]:
result.papers.head()

In [ ]:
result.references.head()

## Mode B — progressive, one stage at a time

Each stage method writes its output to `OUT_DIR` and reads its input from disk if you call it with no arguments. So you can stop after any cell, eyeball the artifact, edit it by hand if you want, and only continue when you're happy.

The pipeline layout under `OUT_DIR`:

```
out/
  markdown/                 # stage 1: docling output, one .md per PDF
  metadata/<paper>.json     # stage 2: per-paper Gemini metadata
  references/<paper>.json   # stage 3: per-paper Gemini references
  papers.csv                # stage 2 aggregated
  references_raw.csv        # stage 3 aggregated
  references.csv            # stage 4 deduplicated
  citation_graph.csv        # stage 4 edges
```

Re-running a stage reuses any per-paper cache files it already wrote, so iterating is cheap.

### Stage 1 — PDFs ➜ markdown

`docling` is slow on first run; cached markdown is reused on subsequent calls. Pass `overwrite_markdown=True` on the `Pipeline` constructor if you need to force a re-conversion.

In [ ]:
markdown_paths = pipe.convert_pdfs()
print(f"{len(markdown_paths)} markdown files in {pipe.layout.markdown_dir}")
for p in markdown_paths[:5]:
    print(" ", p.name)

In [ ]:
# Spot-check the first markdown file before continuing.
if markdown_paths:
    print(markdown_paths[0].read_text()[:1500])

### Stage 2 — markdown ➜ paper metadata

Calls Gemini once per paper. Per-paper JSON is cached under `OUT_DIR/metadata/`, so re-running this cell after editing a single cached JSON file will pick up your edits and rewrite `papers.csv`.

In [ ]:
papers = pipe.extract_paper_metadata()
papers.head()

### Stage 3 — markdown ➜ per-paper references

Also Gemini-driven, also cached per paper under `OUT_DIR/references/`. Output is the long-format `references_raw.csv` with one row per (citing_paper, raw_reference).

In [ ]:
raw_refs = pipe.extract_paper_references()
print(f"{len(raw_refs)} raw references across {raw_refs['citing_id'].nunique()} papers")
raw_refs.head()

### Stage 4 — fuzzy dedup + citation graph

Pure-Python; no network. The dedup config (weights, threshold, year window) is what you'll most often want to tweak between iterations — pass a custom `DedupConfig` to `Pipeline(..., dedup_config=...)` and rerun this cell.

In [ ]:
references, graph = pipe.deduplicate()
print(f"{len(raw_refs)} raw -> {len(references)} unique references, {len(graph)} edges")
references.head()

### Stage 5 — optional CrossRef / OpenAlex enrichment

Skip unless you installed the `[crossref]` extra and want DOIs filled in. Construct a separate `Pipeline` with `enrich=True` (or set the flag on `pipe`) before calling.

In [ ]:
# pipe_enriched = Pipeline(pdf_dir=PDF_DIR, out_dir=OUT_DIR, enrich=True)
# references = pipe_enriched.maybe_enrich()
# references.head()

## Analysis — top-cited references

Works against either mode's output (the artifacts on disk are the same).
If you only ran Mode B, load the CSVs from disk first.

In [ ]:
# Load from disk so this cell works regardless of which mode you ran above.
references = pd.read_csv(pipe.layout.references_csv, index_col="id")
graph = pd.read_csv(pipe.layout.graph_csv)

top = (
    graph.groupby("cited_id")
    .size()
    .sort_values(ascending=False)
    .head(10)
)
references.loc[top.index].assign(citations=top)